# LLM Fiscal and Monetary Policy Classification Demo

Demonstrates fiscal and monetary policy classification prompts with 4 variants each: simple, with_definitions, chain_of_thought, and few_shot.

In [16]:
# Setup
from dotenv import load_dotenv
load_dotenv('../../.env')
import os, sys
from typing import List  # needed for helper signatures
sys.path.insert(0, '../../libs')
sys.path.insert(0, '../../src/Traction/prompts')
sys.path.insert(0, '../../src/Traction')

from llm_factory_openai import LLMAgent
# Use _process_batch_async to process batch_messages
import asyncio
from llm_batch_process_utils import _process_batch_async
from llm_factory_openai import BatchAsyncLLMAgent
from llm_batch_process_utils import _merge_ids_with_responses

from prompt_utils import load_prompt, format_messages
from schema import PROMPT_REGISTRY
from pathlib import Path
import pandas as pd
from llm_batch_process_utils import _build_batch_messages_from_df_flexible
from sklearn.metrics import f1_score, accuracy_score, confusion_matrix, precision_score, recall_score


#### Set up llm

In [2]:
# Initialize LLM Agent
llm_agent = LLMAgent(
    api_key=os.getenv('OPENAI_API_KEY'),
    model='gpt-5-nano',
    temperature=1
)

#### Load prompts templates

In [3]:
# Group prompts by category
prompts = {'monetary_stance': [], 'monetary_agreement': [], 'fiscal_stance': [], 'fiscal_agreement': []}
for key, entry in PROMPT_REGISTRY.items():
    fname = entry["prompt_file"]
    for category in prompts.keys():
        if fname.startswith(category):
            prompts[category].append(key)
            break

print(f"Loaded {sum(len(v) for v in prompts.values())} prompts across {len(prompts)} categories")

Loaded 16 prompts across 4 categories


In [4]:
prompts

{'monetary_stance': ['monetary_stance_simple',
  'monetary_stance_with_definitions',
  'monetary_stance_few_shot',
  'monetary_stance_chain_of_thought'],
 'monetary_agreement': ['monetary_agreement_simple',
  'monetary_agreement_with_definitions',
  'monetary_agreement_few_shot',
  'monetary_agreement_chain_of_thought'],
 'fiscal_stance': ['fiscal_stance_simple',
  'fiscal_stance_with_definitions',
  'fiscal_stance_few_shot',
  'fiscal_stance_chain_of_thought'],
 'fiscal_agreement': ['fiscal_agreement_simple',
  'fiscal_agreement_with_definitions',
  'fiscal_agreement_few_shot',
  'fiscal_agreement_chain_of_thought']}

## Load data

In [6]:
# Load inference data instead of finetuning data

data_dir = Path('/data/home/xiong/data/Fund/CSR/Tractions/output/')
df_inference = pd.read_csv(data_dir / 'document_by_type_sector.csv')
print(f"Loaded inference data with shape: {df_inference.shape}")
print(f"Columns: {list(df_inference.columns)}")


Loaded inference data with shape: (10627, 7)
Columns: ['Print ISBN', 'type', 'topic', 'text', 'country_code', 'country', 'year']


#### defind task parameters 

In [76]:

topic_col = 'topic'
id_col = 'Print ISBN'
type_col = 'type'
text_col = 'text'
country_col = 'country'
year_col = 'year'

# expected column names after pivot
DOMAIN = 'fiscal'
STAFF_COL = 'staff'
AUTHORITY_COL = 'buff'

TASK = 'stance'
PROMPT_STRATEGY = 'few_shot'


In [77]:
def _filter_domain(df: pd.DataFrame, domain: str, topic_col: str ) -> pd.DataFrame:
    """Filter by domain if topic column exists; otherwise return df unchanged."""
    if topic_col in df.columns:
        dom = 'monetary' if domain == 'monetary' else 'fiscal'
        return df[df[topic_col].str.contains(dom, case=False, na=False)].copy()
    return df

def _pivot_agreement_rows(df: pd.DataFrame, *, id_cols: List[str], type_col: str, text_col: str) -> pd.DataFrame:
    """Pivot long rows (types staff/buff) into wide with staff/authority columns.

    Expects `type_col` values to include 'staff' and 'buff' (authority).
    Keeps unique keys by `id_cols` and `topic` if present among id_cols.
    """
    if not set([type_col, text_col]).issubset(df.columns):
        missing = [c for c in [type_col, text_col] if c not in df.columns]
        raise ValueError(f"Missing required columns: {missing}")

    # Normalize type names
    df = df.copy()
    df[type_col] = df[type_col].str.strip().str.lower()
    # Pivot using original role labels ('staff' and 'buff') without renaming
    role_col = type_col
    wide = df.pivot_table(index=id_cols, 
                          columns=role_col, 
                          values=text_col, 
                          aggfunc=lambda x: ' '.join(x))
    wide = wide.reset_index()
    # Ensure expected columns exist
    for col in [STAFF_COL, AUTHORITY_COL]:
        if col not in wide.columns:
            wide[col] = None
    return wide

In [78]:
# Demo: Build batch messages for monetary agreement using flexible function
prompt_key = f"{DOMAIN}_{TASK}_{PROMPT_STRATEGY}"
registry_entry = PROMPT_REGISTRY[prompt_key]
prompt_file = registry_entry["prompt_file"]
prompt_dir = Path('../../src/Traction/prompts')
# Load prompt template
prompt_template = load_prompt(prompt_dir / prompt_file).sections
response_model = registry_entry["response_model"]

In [79]:
df_inference.head()

,Print ISBN,type,topic,text,country_code,country,year
0,9781475517408,buff,Fiscal,17. The authorities noted that fiscal policies...,QAT,Qatar,2015.0
1,9781475517408,buff,Financial,8. The authorities have made significant progr...,QAT,Qatar,2015.0
2,9781475517408,buff,Real,1. Qatar is implementing an ambitious diversif...,QAT,Qatar,2015.0
3,9781475517408,staff,Monetary,6. Monetary and credit conditions remain accom...,QAT,Qatar,2015.0
4,9781475517408,staff,Fiscal,5. The budget continues to post large surpluse...,QAT,Qatar,2015.0


In [80]:
df = _filter_domain(df_inference, DOMAIN, topic_col)
id_cols = [c for c in [id_col, topic_col, country_col, year_col] if c in df.columns]
df_wide = _pivot_agreement_rows(df, id_cols=id_cols, type_col=type_col, text_col=text_col)
df_wide = df_wide.reset_index(drop=True)
df_wide['id'] = df_wide.index.astype(str)
df['id'] = df.index.astype(str)

In [81]:
df.head()

,Print ISBN,type,topic,text,country_code,country,year,id
0,9781475517408,buff,Fiscal,17. The authorities noted that fiscal policies...,QAT,Qatar,2015.0,0
4,9781475517408,staff,Fiscal,5. The budget continues to post large surpluse...,QAT,Qatar,2015.0,4
10,9781475518375,staff,Fiscal,3. Traction. The authorities have made progres...,TON,Tonga,2015.0,10
16,9781475518375,buff,Fiscal,12. The authorities broadly agreed with staff’...,TON,Tonga,2015.0,16
21,9781475519679,staff,Fiscal,"11. Recent developments. In 2014, the fiscal p...",THA,Thailand,2015.0,21


In [82]:
df_wide.head()

type,Print ISBN,topic,country,year,buff,staff,id
0,9781475515039,Fiscal,Uruguay,2015.0,27. The consolidation effort is focused on enh...,8. The medium-term budget for the new governme...,0
1,9781475517231,Fiscal,United Kingdom,2016.0,33. The authorities viewed their fiscal plans ...,Part of the estimated current account gap (1.1...,1
2,9781475517408,Fiscal,Qatar,2015.0,17. The authorities noted that fiscal policies...,5. The budget continues to post large surpluse...,2
3,9781475517675,Fiscal,Ireland,2016.0,20. The government reiterated its broad object...,2. The newly-elected government needs to maint...,3
4,9781475517682,Fiscal,Jamaica,2016.0,Authorities are cognizant of the gains achieve...,1. The Jamaica Labor Party (JLP) won the gener...,4


In [83]:
# Define column mapping: template placeholder -> dataframe column
# Template uses: {COUNTRY}, {YEAR}, {STAFF_TEXT}, {AUTHORITY_TEXT}
# DataFrame has: 'country', 'year', 'staff', 'buff'
column_mapping_agreement = {
    'COUNTRY': 'country',
    'YEAR': 'year',
    'STAFF_TEXT': 'staff',
    'AUTHORITY_TEXT': 'buff'
}

column_mapping_stance = {
    'COUNTRY': 'country',
    'YEAR': 'year',
    'TEXT_AUTHOR': 'type',
    'TEXT': 'text'
}


In [84]:
## select data format based on prompt inputs needed 
sample_df = df_wide if TASK == 'agreement' else df
column_mapping = column_mapping_agreement if TASK == 'agreement' else column_mapping_stance 

In [85]:
# Build batch messages from first 5 rows
batch_messages, batch_ids = _build_batch_messages_from_df_flexible(
    sample_df,
    prompt_template,
    column_mapping=column_mapping,
    id_column='id',  # use the id we just created
    max_text_length=8000
)

In [90]:
print(f"Generated {len(batch_messages)} batch messages")
print(f"Sample IDs: {batch_ids}")
print(f"\nFirst message structure:")
print(f"Number of message parts: {len(batch_messages[0])}")
for i, msg in enumerate(batch_messages[30][:4]):  # Show first 2 parts
    print(f"\nPart {i+1} - Role: {msg.get('role', 'N/A')}")
    content = msg.get('content', '')
    print(f"{content[:1000]}...")


Generated 1953 batch messages
Sample IDs: ['0', '4', '10', '16', '21', '27', '33', '38', '42', '48', '54', '60', '66', '72', '75', '81', '86', '91', '97', '101', '107', '113', '118', '122', '126', '131', '137', '143', '149', '154', '158', '163', '168', '174', '180', '186', '192', '197', '202', '207', '213', '219', '224', '230', '236', '242', '247', '252', '258', '264', '270', '275', '281', '287', '293', '299', '304', '309', '314', '320', '325', '330', '334', '340', '345', '350', '353', '358', '361', '366', '370', '376', '381', '386', '392', '398', '404', '410', '415', '421', '426', '431', '437', '443', '448', '454', '458', '463', '469', '475', '481', '487', '492', '498', '504', '510', '516', '522', '527', '532', '536', '541', '546', '551', '556', '562', '568', '573', '578', '584', '590', '596', '602', '608', '614', '620', '626', '632', '637', '642', '648', '654', '659', '665', '671', '676', '681', '687', '692', '697', '702', '707', '713', '719', '724', '730', '736', '742', '746', '752'

#### Try one example

In [91]:
try:
    response = llm_agent.get_response_content(batch_messages[15], reasoning_effort='low', response_format=response_model)
    print(response)
except Exception as e:
    print(f"Error: {e}")

stance_current='moderately contractionary' stance_future='tightening bias'


In [92]:
batch_agent = BatchAsyncLLMAgent(
    api_key=os.getenv('OPENAI_API_KEY'),
    model='gpt-5',
    temperature=1
)


In [93]:
# Process a small subset to demo
subset_messages = batch_messages[:10]
subset_ids = batch_ids[:10]

async def run_batch():
    results = await _process_batch_async(
        batch_agent,
        subset_messages,
        response_model=registry_entry["response_model"],
        batch_size=8,
        max_completion_tokens=2000,
        reasoning_effort='low'
    )
    return results

results = asyncio.run(run_batch())

Processing batches: 100%|██████████| 10/10 [00:13<00:00,  1.37s/msg]


In [72]:
# Show merged outputs (id + parsed response)
merged = _merge_ids_with_responses(subset_ids, results)
print(f"Processed {len(merged)} responses")
print(merged[0])

Processed 10 responses
{'stance_current': 'accommodative', 'stance_future': 'tightening bias', 'id': '3'}


In [73]:
# Convert merged results to DataFrame
df_merged = pd.DataFrame(merged)
# Add _llm suffix to column names from the LLM results
llm_columns = [col for col in df_merged.columns if col not in ['id']]
rename_dict = {col: f"{col}_llm" for col in llm_columns}
df_merged = df_merged.rename(columns=rename_dict)
df_merged

,stance_current_llm,stance_future_llm,id
0,accommodative,tightening bias,3
1,accommodative,no change,9
2,unclear,unclear,15
3,accommodative,loosening bias,20
4,accommodative,no change,26
5,accommodative,no change,32
6,accommodative,loosening,41
7,accommodative,loosening bias,47
8,accommodative,loosening bias,53
9,accommodative,no change,59


In [74]:
sample_df

,Print ISBN,type,topic,text,country_code,country,year,id
3,9781475517408,staff,Monetary,6. Monetary and credit conditions remain accom...,QAT,Qatar,2015.0,3
9,9781475518375,staff,Monetary,17. Tonga has adopted an informal inflation re...,TON,Tonga,2015.0,9
15,9781475518375,buff,Monetary,19. The authorities agreed with staff’s views ...,TON,Tonga,2015.0,15
20,9781475519679,staff,Monetary,21. Policy stance. While the current monetary ...,THA,Thailand,2015.0,20
26,9781475519679,buff,Monetary,23. Authorities’ views. The authorities viewed...,THA,Thailand,2015.0,26
...,...,...,...,...,...,...,...,...
10599,9798400271380,buff,Monetary,31. The authorities broadly agreed with staff’...,AGO,Angola,2023.0,10599
10605,9798400271878,staff,Monetary,22. Ample bank liquidity and negative real sho...,DZA,Algeria,2023.0,10605
10611,9798400271878,buff,Monetary,29. The BA is carefully following inflationary...,DZA,Algeria,2023.0,10611
10616,9798400279096,buff,Monetary,12. To curtail monetary growth and thaw down i...,SSD,"South Sudan, Republic of",2023.0,10616


In [49]:
# Fix the merge issue by converting id column types to match
df_merged['id'] = df_merged['id'].astype(str)
df_merged = df_merged.merge(sample_df, left_on='id', right_on='id', how='left')
print(len(df_merged))
df_merged.head()

10


,agreement_llm,disagreement_areas_llm,id,Print ISBN_x,topic_x,country_x,year_x,buff_x,staff_x,Print ISBN_y,topic_y,country_y,year_y,buff_y,staff_y
0,disagreement exists,Monetary Policy Framework; Policy Assessment,0,9781475515039,Monetary,Uruguay,2015.0,22. Authorities views on inflation and monetar...,"4. Despite the cooling economy, inflation rema...",9781475515039,Monetary,Uruguay,2015.0,22. Authorities views on inflation and monetar...,"4. Despite the cooling economy, inflation rema..."
1,mostly agree,,1,9781475517231,Monetary,United Kingdom,2016.0,26. Monetary policy settings were viewed as ap...,Credit conditions continue to turn more expans...,9781475517231,Monetary,United Kingdom,2016.0,26. Monetary policy settings were viewed as ap...,Credit conditions continue to turn more expans...
2,irrelevant,,2,9781475517408,Monetary,Qatar,2015.0,NaN,6. Monetary and credit conditions remain accom...,9781475517408,Monetary,Qatar,2015.0,NaN,6. Monetary and credit conditions remain accom...
3,mostly agree,,3,9781475517682,Monetary,Jamaica,2016.0,They agree with the need to further refine the...,18. Monetary policy should be guided by an ove...,9781475517682,Monetary,Jamaica,2016.0,They agree with the need to further refine the...,18. Monetary policy should be guided by an ove...
4,mostly agree,,4,9781475517927,Monetary,Hungary,2016.0,Monetary policy has been accommodative. After ...,4. Inflationary pressures have remained subdue...,9781475517927,Monetary,Hungary,2016.0,Monetary policy has been accommodative. After ...,4. Inflationary pressures have remained subdue...
